In [2]:
import pandas as pd
import numpy as np

In [3]:
# Specify the location of the cleaned dataset created during Phase 2
file_path = "../data/processed/student_performance_cleaned.csv"

# Load the CSV file into a Pandas DataFrame
df = pd.read_csv(file_path)

# Display the dataset dimensions (number of rows and columns)
print("Dataset shape:", df.shape)

# Display the first five records to inspect the loaded dataset
df.head()

Dataset shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [4]:
# Group students into three categories based on their final grades (G3)
# Low: 0–9, Medium: 10–14, High: 15–20
# These boundaries are project-defined assumptions

df["Grade_Category"] = pd.cut(
    df["G3"],
    bins=[-1, 9, 14, 20],
    labels=["Low", "Medium", "High"]
)

# Display final grades alongside their assigned categories
df[["G3", "Grade_Category"]].head(10)

,G3,Grade_Category
0,6,Low
1,6,Low
2,10,Medium
3,15,High
4,10,Medium
5,15,High
6,11,Medium
7,6,Low
8,19,High
9,15,High


In [5]:
# Classify students based on their number of absences.
# Students with 10 or fewer absences are Regular; others are Irregular.

df["Attendance_Status"] = np.where(
    df["absences"] <= 10,
    "Regular",
    "Irregular"
)

# Inspect the attendance classification
df[["absences", "Attendance_Status"]].head(10)

,absences,Attendance_Status
0,6,Regular
1,4,Regular
2,10,Regular
3,2,Regular
4,4,Regular
5,10,Regular
6,0,Regular
7,6,Regular
8,0,Regular
9,0,Regular


In [6]:
# Combine weekly study time and past failures into a single score.
# Higher scores indicate more study time and fewer previous failures.

df["Study_Load"] = df["studytime"] - df["failures"]

# Inspect the calculated Study Load values
df[["studytime", "failures", "Study_Load"]].head(10)

,studytime,failures,Study_Load
0,2,0,2
1,2,0,2
2,2,3,-1
3,3,0,3
4,2,0,2
5,2,0,2
6,2,0,2
7,2,0,2
8,2,0,2
9,2,0,2


In [7]:
# Classify students using the approved passing threshold of 10.
# This is a project-defined threshold, not one specified in the assessment brief.

df["Pass_Fail"] = np.where(
    df["G3"] >= 10,
    "Pass",
    "Fail"
)

# Compare final grades with their corresponding target labels
df[["G3", "Pass_Fail"]].head(10)

,G3,Pass_Fail
0,6,Fail
1,6,Fail
2,10,Pass
3,15,Pass
4,10,Pass
5,15,Pass
6,11,Pass
7,6,Fail
8,19,Pass
9,15,Pass


In [8]:
# Examine the distribution of the engineered categorical features.

print("Grade Category Distribution:")
print(df["Grade_Category"].value_counts().sort_index())

print("\nAttendance Status Distribution:")
print(df["Attendance_Status"].value_counts())

print("\nStudy Load Distribution:")
print(df["Study_Load"].value_counts().sort_index())

print("\nPass/Fail Distribution:")
print(df["Pass_Fail"].value_counts())

Grade Category Distribution:
Grade_Category
Low       130
Medium    192
High       73
Name: count, dtype: int64

Attendance Status Distribution:
Attendance_Status
Regular      329
Irregular     66
Name: count, dtype: int64

Study Load Distribution:
Study_Load
-2      9
-1     13
 0     23
 1    104
 2    165
 3     55
 4     26
Name: count, dtype: int64

Pass/Fail Distribution:
Pass_Fail
Pass    265
Fail    130
Name: count, dtype: int64


In [9]:
# Check the dimensions of the feature-engineered dataset.
print("Dataset shape:", df.shape)

# Check for missing values across all columns.
print("\nTotal missing values:", df.isnull().sum().sum())

# Display the first five records, including the new features.
display(df.head())

# Confirm the data types of the engineered columns.
print("\nEngineered feature data types:")
print(df[["Grade_Category", "Attendance_Status",
          "Study_Load", "Pass_Fail"]].dtypes)

Dataset shape: (395, 37)

Total missing values: 0


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,Walc,health,absences,G1,G2,G3,Grade_Category,Attendance_Status,Study_Load,Pass_Fail
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,1,3,6,5,6,6,Low,Regular,2,Fail
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,1,3,4,5,5,6,Low,Regular,2,Fail
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,3,10,7,8,10,Medium,Regular,-1,Pass
3,GP,F,15,U,GT3,T,4,2,health,services,...,1,5,2,15,14,15,High,Regular,3,Pass
4,GP,F,16,U,GT3,T,3,3,other,other,...,2,5,4,6,10,10,Medium,Regular,2,Pass



Engineered feature data types:
Grade_Category       category
Attendance_Status         str
Study_Load              int64
Pass_Fail                 str
dtype: object


In [10]:
# Define the output path for the feature-engineered dataset.
output_path = "../data/processed/student_performance_featured.csv"

# Save the dataset without including the DataFrame index.
df.to_csv(output_path, index=False)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.


## Feature Engineering Summary

Four new features were created to enrich the student performance dataset:

1. **Grade Category:** Final grades were grouped into Low (0–9), Medium (10–14), and High (15–20).
2. **Attendance Status:** Students with 10 or fewer absences were classified as Regular, while those with more than 10 were classified as Irregular.
3. **Study Load:** A composite score was calculated by subtracting the number of past failures from the weekly study-time category.
4. **Pass/Fail:** Students with a final grade of 10 or above were classified as Pass; those below 10 were classified as Fail.

These definitions are project-defined operational assumptions.

The resulting dataset contains 395 records and 37 columns, with no missing values. The engineered dataset was saved as `student_performance_featured.csv`.

**Modelling consideration:** Grade Category and G3 are derived from the final grade and will be excluded from model predictors to prevent target leakage. Pass_Fail will serve as the classification target.